<a href="https://colab.research.google.com/github/Hassanmufezshaikh/AI-Agents/blob/main/human_in_loop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --upgrade google-adk google-genai

  Using cached google_genai-2.7.0-py3-none-any.whl.metadata (52 kB)


In [ ]:
import google.adk
print(google.adk.__version__)

2.1.0


In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64
print(" Tunnel Components imported successfully")



 Tunnel Components imported successfully


In [ ]:
import os
from google.colab import userdata
userdata.get('gemeni')

try:
  GOOGLE_API_KEY = userdata.get('gemeni')
  os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
  print("Gemini API Key Setup Complete")
except Exception as e :
  print("Authencation Error: Please add 'GEMENI_API_KEY' to your kaggale secrets, Details : {e}")

Gemini API Key Setup Complete


In [ ]:
from google.adk.agents import (
    Agent,
    SequentialAgent,
    ParallelAgent,
    LoopAgent
)

from google.adk.models.google_llm import Gemini
from google.adk.runners import InMemoryRunner
from google.adk.tools import google_search, AgentTool, FunctionTool
from google.genai import types

print("ADK components imported successfully.")

ADK components imported successfully.


In [ ]:
import google.adk.tools as tools

print(dir(tools))

['APIHubToolset', 'AgentTool', 'AgentTool', 'Any', 'ApiRegistry', 'AuthToolArguments', 'BaseTool', 'DiscoveryEngineSearchTool', 'ExampleTool', 'FunctionTool', 'FunctionTool', 'LongRunningFunctionTool', 'MCPToolset', 'McpToolset', 'SearchResultMode', 'TYPE_CHECKING', 'ToolContext', 'TransferToAgentTool', 'VertexAiSearchTool', '_LAZY_MAPPING', '__all__', '__builtins__', '__cached__', '__dir__', '__doc__', '__file__', '__getattr__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '_automatic_function_calling_util', '_forwarding_artifact_service', '_function_parameter_parse_util', '_function_tool_declarations', 'agent_tool', 'base_tool', 'base_toolset', 'computer_use', 'enterprise_web_search', 'exit_loop', 'function_tool', 'get_user_choice', 'google_maps_grounding', 'google_search', 'google_search', 'google_search_tool', 'importlib', 'load_artifacts', 'load_memory', 'logging', 'preload_memory', 'set_model_response_tool', 'sys', 'tool_configs', 'tool_confirmation', 'tool_co

In [ ]:
from google.genai import types

retry_config=types.HttpRetryOptions(
attempts=5, # Maximum retry attempts
exp_base=7, # Delay multiplier
initial_delay=1, # Initial delay before first retry (in seconds)
http_status_codes=[429, 500, 503, 504]
)

In [ ]:
# ============================================================
# HUMAN-IN-THE-LOOP TOOL APPROVAL FUNCTION
# Google ADK + Google Colab Compatible
# ============================================================

from google.adk.tools import ToolContext

# Orders above this need human approval
LARGE_ORDER_THRESHOLD = 5


def place_shipping_order(
    num_containers: int,
    destination: str,
    tool_context: ToolContext
) -> dict:
    """
    Places a shipping order.

    Small orders are auto-approved.
    Large orders require human approval.

    This function supports:
    1. Initial call
    2. Pause for approval
    3. Resume after approval/rejection
    """

    # ============================================================
    # SCENARIO 1:
    # Small orders -> Auto Approved
    # ============================================================

    if num_containers <= LARGE_ORDER_THRESHOLD:
        return {
            "status": "approved",
            "order_id": f"ORD-{num_containers}-AUTO",
            "num_containers": num_containers,
            "destination": destination,
            "message": (
                f"Order auto-approved for "
                f"{num_containers} containers to {destination}"
            )
        }

    # ============================================================
    # SCENARIO 2:
    # First time tool call for LARGE order
    # Request human approval -> PAUSE
    # ============================================================

    if not tool_context.tool_confirmation:

        tool_context.request_confirmation(
            hint=(
                f"Large order detected.\n"
                f"{num_containers} containers to {destination}.\n"
                f"Do you want to approve this shipment?"
            ),
            payload={
                "num_containers": num_containers,
                "destination": destination
            }
        )

        return {
            "status": "pending_approval",
            "message": (
                "Human approval required before processing order."
            )
        }

    # ============================================================
    # SCENARIO 3:
    # Tool called AGAIN after pause
    # RESUME here
    # Handle approval/rejection
    # ============================================================

    if tool_context.tool_confirmation.confirmed:

        return {
            "status": "approved",
            "order_id": f"ORD-{num_containers}-HUMAN",
            "num_containers": num_containers,
            "destination": destination,
            "message": (
                f"Human approved shipment of "
                f"{num_containers} containers to {destination}"
            )
        }

    else:

        return {
            "status": "rejected",
            "num_containers": num_containers,
            "destination": destination,
            "message": (
                f"Human rejected shipment of "
                f"{num_containers} containers to {destination}"
            )
        }


print("place_shipping_order function created successfully.")

place_shipping_order function created successfully.


In [ ]:
model = Gemini(model_id='gemini-2.0-flash')

# In ADK, FunctionTool automatically uses the function's docstring as the description.
shipping_tool = FunctionTool(
    func=place_shipping_order
)

shipping_agent = Agent(
    name="ShippingAgent",
    model=model,
    instruction="You are a logistics assistant. Use the place_shipping_order tool to handle customer requests.",
    tools=[shipping_tool]
)

# InMemoryRunner uses the keyword 'agent'
runner = InMemoryRunner(agent=shipping_agent)

In [ ]:
import inspect
from google.adk.runners import InMemoryRunner

# Inspect the run method signature to see exactly what arguments it expects
sig = inspect.signature(InMemoryRunner.run)
print(f"InMemoryRunner.run signature: {sig}")

# Also check the docstring for details
print("\nMethod Documentation:")
print(InMemoryRunner.run.__doc__)

InMemoryRunner.run signature: (self, *, user_id: 'str', session_id: 'str', new_message: 'types.Content', run_config: 'Optional[RunConfig]' = None) -> 'Generator[Event, None, None]'

Method Documentation:
Runs the agent.

    NOTE:
      This sync interface is only for local testing and convenience purpose.
      Consider using `run_async` for production usage.

    If event compaction is enabled in the App configuration, it will be
    performed after all agent events for the current invocation have been
    yielded. The generator will only finish iterating after event
    compaction is complete.

    Args:
      user_id: The user ID of the session.
      session_id: The session ID of the session.
      new_message: A new message to append to the session.
      run_config: The run config for the agent.

    Yields:
      The events generated by the agent.
    
